# AQI Forecasting Regression

## Problem Statement

Predict next-hour European Air Quality Index (`European_AQI`).

## Why This Variation Was Chosen

This version forecasts overall air-quality severity rather than one pollutant, which is useful for city-level severity planning and interpretation.

## Models Planned

This notebook compares persistence baseline, Ridge Regression, Random Forest Regressor, HistGradientBoostingRegressor, and optional PyTorch GRU sequence regressor.

## Assignment Fit

The assignment requires a challenging dataset, preprocessing, at least three models, model validation, critical comparison, and recommendations. This notebook is standalone and shows every step inline: dataset explanation, EDA, preprocessing, feature engineering, chronological splitting, model training, testing, evaluation, interpretation, unsupervised profiling, and conclusion.


## Dataset Column Dictionary

| Column | Meaning | ML role |
| --- | --- | --- |
| `Timestamp` | Hourly observation time. | Parse to datetime; derive hour, weekday, month, cyclic features, lags, and chronological splits. |
| `City` | City where the observation belongs. | Categorical location feature; use one global model with city information. |
| `Latitude` / `Longitude` | City coordinates. | Numeric location features; constant within each city. |
| `PM10_ug_m3` | Coarse particulate matter concentration. | Predictor; contains light missingness that must be imputed. |
| `PM2_5_ug_m3` | Fine particulate matter concentration. | Strong pollutant predictor and forecasting target candidate. |
| `Carbon_Monoxide_ug_m3` | Carbon monoxide concentration. | Combustion/traffic-related predictor. |
| `Nitrogen_Dioxide_ug_m3` | Nitrogen dioxide concentration. | Traffic/industrial pollution predictor. |
| `Ozone_ug_m3` | Ground-level ozone concentration. | Pollutant affected by sunlight and atmospheric reactions. |
| `Dust_ug_m3` | Airborne dust concentration. | Heavy-tailed pollution predictor. |
| `UV_Index` | Ultraviolet intensity. | Environmental/time-of-day related predictor. |
| `European_AQI` | Combined European Air Quality Index. | Regression/forecasting target candidate; potential leakage when predicting `Hazardous_Event`. |
| `Hazardous_Event` | Binary hazardous-event flag. | Classification target candidate. |

## Setup and Runtime Controls

The notebook defaults to heavier experiments when CUDA/PyTorch is available. Set `RUN_BALANCED_BACKUP=True` or environment variable `AML_BALANCED_BACKUP=1` for a faster CPU-safe path. PyTorch installation is deliberate, not automatic, unless `INSTALL_TORCH_IF_MISSING=True` is set.

In [ ]:
from pathlib import Path
import os, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, precision_recall_curve, precision_score, r2_score, recall_score, roc_auc_score
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except Exception as exc:
    torch = nn = DataLoader = TensorDataset = None
    TORCH_AVAILABLE = False
    TORCH_IMPORT_ERROR = exc
warnings.filterwarnings("ignore", category=ConvergenceWarning)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 120)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.parent != PROJECT_ROOT and not (PROJECT_ROOT / "AGENTS.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "Datasets" / "Global Urban Air Quality & Pollution Time-Series" / "global_urban_smog_pm25_hourly.csv"

# Heavy mode is the default when PyTorch CUDA is installed. Use backup/smoke mode for faster CPU-safe runs.
RUN_BALANCED_BACKUP = os.environ.get("AML_BALANCED_BACKUP", "0") == "1"
FAST_SMOKE_TEST = os.environ.get("AML_FAST_SMOKE", "0") == "1"
INSTALL_TORCH_IF_MISSING = False
if FAST_SMOKE_TEST:
    RUN_BALANCED_BACKUP = True
if INSTALL_TORCH_IF_MISSING and not TORCH_AVAILABLE:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu128"])
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
CUDA_AVAILABLE = bool(TORCH_AVAILABLE and torch.cuda.is_available())
DEVICE = "cuda" if CUDA_AVAILABLE and not RUN_BALANCED_BACKUP else "cpu"
display(pd.DataFrame([dict(project_root=str(PROJECT_ROOT), dataset_found=DATA_PATH.exists(), torch_available=TORCH_AVAILABLE, cuda_available=CUDA_AVAILABLE, pytorch_device=DEVICE, balanced_backup_mode=RUN_BALANCED_BACKUP, fast_smoke_test=FAST_SMOKE_TEST)]))
if not TORCH_AVAILABLE:
    display(Markdown("**PyTorch is not installed.** The notebook still runs with scikit-learn. For GPU experiments, run: `.venv\\Scripts\\python.exe -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128`."))
elif CUDA_AVAILABLE:
    display(Markdown(f"**CUDA detected:** `{torch.cuda.get_device_name(0)}`."))
else:
    display(Markdown("**PyTorch is installed but CUDA is not available.**"))

In [ ]:
VARIATION_KEY = "aqi_forecasting_regression"
VARIATION_TITLE = "AQI Forecasting Regression"
TASK_TYPE = "regression"
TARGET_SOURCE = "European_AQI"
TARGET_LABEL = "European_AQI_t_plus_1h"
HORIZON_HOURS = 1
INCLUDE_EUROPEAN_AQI = True
DEEP_MODEL_KIND = "gru_regressor"
display(pd.DataFrame([dict(variation=VARIATION_TITLE, task_type=TASK_TYPE, target_source=TARGET_SOURCE, target_label=TARGET_LABEL, horizon_hours=HORIZON_HOURS, include_european_aqi_feature=INCLUDE_EUROPEAN_AQI, model_selection_focus="validation MAE")]))

## Reusable Functions Defined Inside This Notebook

All helper functions are included here so this notebook does not depend on any shared project pipeline file.

In [ ]:
REQUIRED_COLUMNS = ["Timestamp","City","Latitude","Longitude","PM10_ug_m3","PM2_5_ug_m3","Carbon_Monoxide_ug_m3","Nitrogen_Dioxide_ug_m3","Ozone_ug_m3","Dust_ug_m3","UV_Index","European_AQI","Hazardous_Event"]
POLLUTION_COLUMNS = ["PM10_ug_m3","PM2_5_ug_m3","Carbon_Monoxide_ug_m3","Nitrogen_Dioxide_ug_m3","Ozone_ug_m3","Dust_ug_m3","UV_Index","European_AQI"]
NO_AQI_POLLUTION_COLUMNS = [c for c in POLLUTION_COLUMNS if c != "European_AQI"]
TIME_FEATURES = ["hour","weekday","month","dayofyear","is_weekend","hour_sin","hour_cos","dayofyear_sin","dayofyear_cos"]
LOCATION_FEATURES = ["Latitude","Longitude"]
CATEGORICAL_FEATURES = ["City"]
LAG_STEPS = [1, 3, 6, 24]
ROLLING_WINDOWS = [3, 6, 24]
SEQUENCE_LENGTH = 24
TRAIN_END = pd.Timestamp("2026-03-31 23:00:00")
VALID_START = pd.Timestamp("2026-04-01 00:00:00")
VALID_END = pd.Timestamp("2026-04-30 23:00:00")
TEST_START = pd.Timestamp("2026-05-01 00:00:00")
TEST_END = pd.Timestamp("2026-05-22 23:00:00")

def require_columns(df):
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

def add_time_features(df):
    out = df.copy()
    out["hour"] = out["Timestamp"].dt.hour
    out["weekday"] = out["Timestamp"].dt.weekday
    out["month"] = out["Timestamp"].dt.month
    out["dayofyear"] = out["Timestamp"].dt.dayofyear
    out["is_weekend"] = out["weekday"].isin([5, 6]).astype(int)
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["dayofyear_sin"] = np.sin(2 * np.pi * out["dayofyear"] / 366)
    out["dayofyear_cos"] = np.cos(2 * np.pi * out["dayofyear"] / 366)
    return out

def clean_dataset(df):
    require_columns(df)
    out = df.copy()
    out["Timestamp"] = pd.to_datetime(out["Timestamp"], errors="raise")
    out = out.sort_values(["City", "Timestamp"]).reset_index(drop=True)
    missing_before = int(out["PM10_ug_m3"].isna().sum())
    out["PM10_ug_m3"] = out.groupby("City")["PM10_ug_m3"].transform(lambda s: s.interpolate(method="linear", limit_direction="both"))
    out["PM10_ug_m3"] = out["PM10_ug_m3"].fillna(out["PM10_ug_m3"].median())
    out = add_time_features(out)
    return out, pd.DataFrame([dict(rows=len(out), columns=out.shape[1], cities=out["City"].nunique(), timestamp_min=out["Timestamp"].min(), timestamp_max=out["Timestamp"].max(), pm10_missing_before=missing_before, pm10_missing_after=int(out["PM10_ug_m3"].isna().sum()), hazard_rate=out["Hazardous_Event"].mean())])

def build_model_frame(df, target_source, target_label, horizon_hours, include_european_aqi):
    out = df.copy()
    signal_columns = POLLUTION_COLUMNS if include_european_aqi else NO_AQI_POLLUTION_COLUMNS
    grouped = out.groupby("City", group_keys=False)
    lag_features, rolling_features = [], []
    for column in signal_columns:
        for lag in LAG_STEPS:
            name = f"{column}_lag_{lag}h"
            out[name] = grouped[column].shift(lag)
            lag_features.append(name)
        for window in ROLLING_WINDOWS:
            name = f"{column}_rolling_mean_{window}h"
            out[name] = grouped[column].transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=w).mean())
            rolling_features.append(name)
    if horizon_hours > 0:
        out[target_label] = grouped[target_source].shift(-horizon_hours)
        out["Target_Timestamp"] = grouped["Timestamp"].shift(-horizon_hours)
    else:
        out[target_label] = out[target_source]
        out["Target_Timestamp"] = out["Timestamp"]
    feature_columns = signal_columns + lag_features + rolling_features + TIME_FEATURES + LOCATION_FEATURES + CATEGORICAL_FEATURES
    before = len(out)
    out = out.dropna(subset=feature_columns + [target_label, "Target_Timestamp"]).reset_index(drop=True)
    if TASK_TYPE == "classification":
        out[target_label] = out[target_label].astype(int)
    summary = pd.DataFrame([dict(rows_before_lag_target_drop=before, rows_after_lag_target_drop=len(out), rows_dropped=before-len(out), feature_count=len(feature_columns), uses_european_aqi_as_feature=include_european_aqi, target_label=target_label, horizon_hours=horizon_hours)])
    return out, feature_columns, summary

def chronological_split(model_df, target_label):
    masks = dict(train=model_df["Target_Timestamp"] <= TRAIN_END, valid=model_df["Target_Timestamp"].between(VALID_START, VALID_END), test=model_df["Target_Timestamp"].between(TEST_START, TEST_END))
    return {name: dict(X=model_df.loc[mask].copy(), y=model_df.loc[mask, target_label].copy(), meta=model_df.loc[mask, ["Timestamp", "Target_Timestamp", "City"]].copy()) for name, mask in masks.items()}

def runtime_sample(split, max_rows, task_type):
    X, y, meta = split["X"], split["y"], split["meta"]
    if not FAST_SMOKE_TEST or len(X) <= max_rows:
        return split
    frame = X.assign(__target_for_sample__=y.values)
    if task_type == "classification" and y.nunique() == 2:
        pieces = []
        for _, part in frame.groupby("__target_for_sample__"):
            n = max(1, int(max_rows * len(part) / len(frame)))
            pieces.append(part.sample(n=min(n, len(part)), random_state=RANDOM_STATE))
        idx = pd.concat(pieces).sample(frac=1, random_state=RANDOM_STATE).index[:max_rows]
    else:
        idx = frame.sample(n=max_rows, random_state=RANDOM_STATE).index
    return dict(X=X.loc[idx], y=y.loc[idx], meta=meta.loc[idx])

def make_preprocessor(feature_columns):
    numeric_features = [c for c in feature_columns if c not in CATEGORICAL_FEATURES]
    return ColumnTransformer([
        ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
        ("city", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CATEGORICAL_FEATURES),
    ], remainder="drop", verbose_feature_names_out=False)

def make_pipeline(estimator, feature_columns):
    return Pipeline([("preprocess", make_preprocessor(feature_columns)), ("model", estimator)])

def predict_positive_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X)
        return 1 / (1 + np.exp(-raw))
    return model.predict(X)

def choose_threshold(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    if len(thresholds) == 0:
        return 0.5
    f1_values = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    return float(thresholds[int(np.nanargmax(f1_values))])

def classification_metrics(y_true, scores, threshold):
    pred = (scores >= threshold).astype(int)
    values = dict(accuracy=accuracy_score(y_true, pred), precision=precision_score(y_true, pred, zero_division=0), recall=recall_score(y_true, pred, zero_division=0), f1=f1_score(y_true, pred, zero_division=0), roc_auc=roc_auc_score(y_true, scores) if y_true.nunique() == 2 else np.nan, pr_auc=average_precision_score(y_true, scores) if y_true.nunique() == 2 else np.nan)
    return values, pred

def regression_metrics(y_true, pred):
    return dict(mae=mean_absolute_error(y_true, pred), rmse=math.sqrt(mean_squared_error(y_true, pred)), r2=r2_score(y_true, pred))

def model_specs(task_type):
    balanced = RUN_BALANCED_BACKUP or FAST_SMOKE_TEST
    if task_type == "classification":
        return {
            "Logistic Regression": (LogisticRegression(class_weight="balanced", max_iter=500 if balanced else 1000, random_state=RANDOM_STATE), {"model__C": [0.5, 1.0] if balanced else [0.3, 1.0, 3.0]}),
            "Random Forest": (RandomForestClassifier(n_estimators=60 if balanced else 160, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE), {"model__max_depth": [10] if balanced else [10, 18], "model__min_samples_leaf": [5] if balanced else [2, 8]}),
            "HistGradientBoosting": (HistGradientBoostingClassifier(max_iter=60 if balanced else 180, random_state=RANDOM_STATE), {"model__learning_rate": [0.08] if balanced else [0.05, 0.1], "model__max_leaf_nodes": [31] if balanced else [31, 63]}),
        }
    return {
        "Ridge Regression": (Ridge(random_state=RANDOM_STATE), {"model__alpha": [1.0, 10.0] if balanced else [0.1, 1.0, 10.0]}),
        "Random Forest Regressor": (RandomForestRegressor(n_estimators=60 if balanced else 160, n_jobs=-1, random_state=RANDOM_STATE), {"model__max_depth": [10] if balanced else [10, 18], "model__min_samples_leaf": [5] if balanced else [2, 8]}),
        "HistGradientBoosting Regressor": (HistGradientBoostingRegressor(max_iter=60 if balanced else 180, random_state=RANDOM_STATE), {"model__learning_rate": [0.08] if balanced else [0.05, 0.1], "model__max_leaf_nodes": [31] if balanced else [31, 63]}),
    }

## Data Loading, Cleaning, and EDA

This section loads the raw CSV, checks schema, explains missingness, cleans `PM10_ug_m3`, and explores city/time/pollution patterns with inline tables and plots.

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
require_columns(df_raw)
display(Markdown(f"Loaded raw dataset from `{DATA_PATH}`."))
display(pd.DataFrame([{"rows": len(df_raw), "columns": df_raw.shape[1]}]))
display(df_raw.head())
df_clean, cleaning_summary = clean_dataset(df_raw)
display(Markdown("### Cleaning summary"))
display(cleaning_summary)
schema_summary = pd.DataFrame({"dtype": df_raw.dtypes.astype(str), "missing_before_cleaning": df_raw.isna().sum(), "unique_values": df_raw.nunique(dropna=True)}).reset_index(names="column")
display(Markdown("### Data types and missing values"))
display(schema_summary)
fig, ax = plt.subplots(figsize=(10, 4))
schema_summary.set_index("column")["missing_before_cleaning"].plot(kind="bar", ax=ax)
ax.set_title("Missing values before cleaning")
ax.set_ylabel("Missing rows")
plt.xticks(rotation=45, ha="right")
plt.show()
coverage = pd.DataFrame([dict(start=df_clean["Timestamp"].min(), end=df_clean["Timestamp"].max(), unique_hourly_timestamps=df_clean["Timestamp"].nunique(), cities=df_clean["City"].nunique(), rows_per_city_min=df_clean.groupby("City").size().min(), rows_per_city_max=df_clean.groupby("City").size().max())])
display(Markdown("### Time and city coverage"))
display(coverage)
city_pollution = df_clean.groupby("City").agg(rows=("Timestamp", "size"), pm25_mean=("PM2_5_ug_m3", "mean"), pm10_mean=("PM10_ug_m3", "mean"), aqi_mean=("European_AQI", "mean"), hazard_rate=("Hazardous_Event", "mean")).sort_values("pm25_mean", ascending=False)
display(city_pollution.head(10))
fig, ax = plt.subplots(figsize=(10, 8))
city_pollution["pm25_mean"].sort_values().plot(kind="barh", ax=ax)
ax.set_title("Mean PM2.5 by city")
ax.set_xlabel("Mean PM2.5 (ug/m3)")
plt.show()
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.histplot(df_clean["PM2_5_ug_m3"], bins=80, kde=True, ax=axes[0, 0]); axes[0, 0].set_title("PM2.5 distribution")
sns.histplot(df_clean["European_AQI"], bins=80, kde=True, ax=axes[0, 1]); axes[0, 1].set_title("European AQI distribution")
sns.countplot(data=df_clean, x="Hazardous_Event", ax=axes[1, 0]); axes[1, 0].set_title("Hazardous event class distribution")
hourly_profile = df_clean.groupby("hour")[["PM2_5_ug_m3", "European_AQI", "Hazardous_Event"]].mean()
hourly_profile[["PM2_5_ug_m3", "European_AQI"]].plot(ax=axes[1, 1]); axes[1, 1].set_title("Mean PM2.5 and AQI by hour"); axes[1, 1].set_xlabel("Hour of day")
plt.tight_layout(); plt.show()
corr = df_clean[POLLUTION_COLUMNS + ["Hazardous_Event"]].corr(numeric_only=True)
display(Markdown("### Pollutant correlation"))
display(corr.round(3))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap="vlag", center=0, annot=False, ax=ax)
ax.set_title("Correlation between pollutants, AQI, and hazardous flag")
plt.show()

## Target Construction, Feature Engineering, and Chronological Split

The target is created according to this variation. Lag and rolling features are built within each city so values never cross city boundaries. Splits are chronological by target timestamp.

In [ ]:
model_df, feature_columns, feature_summary = build_model_frame(df_clean, TARGET_SOURCE, TARGET_LABEL, HORIZON_HOURS, INCLUDE_EUROPEAN_AQI)
if TASK_TYPE == "classification" and not INCLUDE_EUROPEAN_AQI:
    assert "European_AQI" not in feature_columns
splits = chronological_split(model_df, TARGET_LABEL)
split_summary = pd.DataFrame([dict(split=name, rows=len(split["X"]), target_start=split["X"]["Target_Timestamp"].min(), target_end=split["X"]["Target_Timestamp"].max(), target_mean=split["y"].mean()) for name, split in splits.items()])
display(Markdown("### Feature engineering summary")); display(feature_summary)
display(Markdown("### Chronological split summary")); display(split_summary)
display(model_df[["Timestamp", "Target_Timestamp", "City", TARGET_LABEL] + feature_columns[:8]].head())
if TASK_TYPE == "classification":
    display(pd.DataFrame([dict(split=name, class_0=int((split["y"] == 0).sum()), class_1=int((split["y"] == 1).sum()), positive_rate=split["y"].mean()) for name, split in splits.items()]))
    fig, ax = plt.subplots(figsize=(7, 4)); sns.countplot(data=model_df, x=TARGET_LABEL, ax=ax); ax.set_title(f"Target distribution: {TARGET_LABEL}"); plt.show()
else:
    display(pd.DataFrame([dict(split=name, mean=split["y"].mean(), median=split["y"].median(), std=split["y"].std(), min=split["y"].min(), max=split["y"].max()) for name, split in splits.items()]))
    fig, ax = plt.subplots(figsize=(9, 4)); sns.histplot(model_df[TARGET_LABEL], bins=80, kde=True, ax=ax); ax.set_title(f"Target distribution: {TARGET_LABEL}"); plt.show()
fig, ax = plt.subplots(figsize=(11, 7))
model_df.groupby("City")[TARGET_LABEL].mean().sort_values().plot(kind="barh", ax=ax)
ax.set_title(f"Mean target value by city: {TARGET_LABEL}")
plt.show()
fig, ax = plt.subplots(figsize=(8, 4))
model_df.groupby("hour")[TARGET_LABEL].mean().plot(marker="o", ax=ax)
ax.set_title(f"Mean target value by hour: {TARGET_LABEL}")
ax.set_xlabel("Hour of day")
plt.show()

## Unsupervised Learning Component

K-Means and PCA profile cities by pollution characteristics, supporting the unsupervised learning outcome and interpretation.

In [ ]:
display(Markdown("This unsupervised section groups cities by pollutant profiles to support interpretation and recommendations."))
profile = df_clean.groupby("City")[NO_AQI_POLLUTION_COLUMNS + ["European_AQI", "Hazardous_Event"]].agg(["mean", "std"])
profile.columns = ["_".join(parts) for parts in profile.columns]
profile = profile.fillna(0)
scaled_profile = StandardScaler().fit_transform(profile)
clusters = KMeans(n_clusters=4, n_init="auto", random_state=RANDOM_STATE).fit_predict(scaled_profile)
coords = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(scaled_profile)
cluster_view = profile.copy()
cluster_view["cluster"] = clusters
cluster_view["pca_1"] = coords[:, 0]
cluster_view["pca_2"] = coords[:, 1]
display(cluster_view.sort_values(["cluster", "European_AQI_mean"])[["cluster", "PM2_5_ug_m3_mean", "PM10_ug_m3_mean", "European_AQI_mean", "Hazardous_Event_mean", "pca_1", "pca_2"]])
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=cluster_view.reset_index(), x="pca_1", y="pca_2", hue="cluster", palette="tab10", s=90, ax=ax)
for city, row in cluster_view.iterrows():
    ax.text(row["pca_1"], row["pca_2"], city, fontsize=8)
ax.set_title("K-Means city pollution profiles visualized with PCA")
plt.show()

## Supervised Model Training and Tuning

This section trains baselines and at least three tuned machine-learning models. Classification models are selected by validation F1; regression models are selected by validation MAE.

In [ ]:
eval_splits = {name: runtime_sample(splits[name], 14000 if name == "train" else 5000, TASK_TYPE) for name in ["train", "valid", "test"]}
X_train, y_train = eval_splits["train"]["X"], eval_splits["train"]["y"]
X_valid, y_valid = eval_splits["valid"]["X"], eval_splits["valid"]["y"]
X_test, y_test, meta_test = eval_splits["test"]["X"], eval_splits["test"]["y"], eval_splits["test"]["meta"]
results, prediction_tables, trained_sklearn_models = [], {}, {}
if TASK_TYPE == "classification":
    majority = int(y_train.mode().iloc[0])
    valid_scores = np.full(len(y_valid), majority, dtype=float); test_scores = np.full(len(y_test), majority, dtype=float)
    valid_values, _ = classification_metrics(y_valid, valid_scores, 0.5); test_values, test_pred = classification_metrics(y_test, test_scores, 0.5)
    run_id = "Majority baseline"
    results.append(dict(run_id=run_id, model="Majority baseline", params={}, threshold=0.5, **{f"valid_{k}": v for k, v in valid_values.items()}, **{f"test_{k}": v for k, v in test_values.items()}))
    prediction_tables[run_id] = meta_test.assign(actual=y_test.values, predicted=test_pred, positive_score=test_scores)
else:
    mean_pred = float(y_train.mean())
    valid_pred = np.full(len(y_valid), mean_pred); test_pred = np.full(len(y_test), mean_pred)
    valid_values = regression_metrics(y_valid, valid_pred); test_values = regression_metrics(y_test, test_pred)
    run_id = "Mean baseline"
    results.append(dict(run_id=run_id, model="Mean baseline", params={"prediction": mean_pred}, **{f"valid_{k}": v for k, v in valid_values.items()}, **{f"test_{k}": v for k, v in test_values.items()}))
    prediction_tables[run_id] = meta_test.assign(actual=y_test.values, predicted=test_pred, error=test_pred-y_test.values)
    if TARGET_SOURCE in X_valid.columns:
        valid_pred = X_valid[TARGET_SOURCE].to_numpy(); test_pred = X_test[TARGET_SOURCE].to_numpy()
        valid_values = regression_metrics(y_valid, valid_pred); test_values = regression_metrics(y_test, test_pred)
        run_id = "Persistence baseline"
        results.append(dict(run_id=run_id, model="Persistence baseline", params={"prediction": f"current {TARGET_SOURCE}"}, **{f"valid_{k}": v for k, v in valid_values.items()}, **{f"test_{k}": v for k, v in test_values.items()}))
        prediction_tables[run_id] = meta_test.assign(actual=y_test.values, predicted=test_pred, error=test_pred-y_test.values)
for model_name, (estimator, grid) in model_specs(TASK_TYPE).items():
    for params in ParameterGrid(grid):
        model = make_pipeline(estimator, feature_columns)
        model.set_params(**params)
        model.fit(X_train[feature_columns], y_train)
        run_id = f"{model_name} #{len(results) + 1}"
        if TASK_TYPE == "classification":
            valid_scores = predict_positive_scores(model, X_valid[feature_columns])
            threshold = choose_threshold(y_valid, valid_scores)
            test_scores = predict_positive_scores(model, X_test[feature_columns])
            valid_values, _ = classification_metrics(y_valid, valid_scores, threshold)
            test_values, test_pred = classification_metrics(y_test, test_scores, threshold)
            row = dict(run_id=run_id, model=model_name, params=params, threshold=threshold, **{f"valid_{k}": v for k, v in valid_values.items()}, **{f"test_{k}": v for k, v in test_values.items()})
            prediction_tables[run_id] = meta_test.assign(actual=y_test.values, predicted=test_pred, positive_score=test_scores)
        else:
            valid_pred = model.predict(X_valid[feature_columns]); test_pred = model.predict(X_test[feature_columns])
            valid_values = regression_metrics(y_valid, valid_pred); test_values = regression_metrics(y_test, test_pred)
            row = dict(run_id=run_id, model=model_name, params=params, **{f"valid_{k}": v for k, v in valid_values.items()}, **{f"test_{k}": v for k, v in test_values.items()})
            prediction_tables[run_id] = meta_test.assign(actual=y_test.values, predicted=test_pred, error=test_pred-y_test.values)
        results.append(row)
        trained_sklearn_models[run_id] = model
results_df = pd.DataFrame(results)
display(results_df.sort_values("valid_f1" if TASK_TYPE == "classification" else "valid_mae", ascending=(TASK_TYPE != "classification")))

## GPU Neural-Network Experiment

This optional section uses PyTorch if available. It does not save model files; it only displays training history and appends inline metrics.

In [ ]:
def split_by_target_timestamp(timestamps):
    return dict(train=timestamps <= TRAIN_END, valid=timestamps.between(VALID_START, VALID_END), test=timestamps.between(TEST_START, TEST_END))

def build_sequence_dataset(df, feature_cols, target_source, horizon_hours, sequence_length):
    sequences, targets, target_timestamps, cities = [], [], [], []
    for city, group in df.sort_values(["City", "Timestamp"]).groupby("City"):
        group = group.reset_index(drop=True)
        features = group[feature_cols].to_numpy(dtype=np.float32)
        target_values = group[target_source].to_numpy(dtype=np.float32)
        timestamps = group["Timestamp"].reset_index(drop=True)
        for end in range(sequence_length - 1, len(group) - horizon_hours):
            target_index = end + horizon_hours
            sequences.append(features[end-sequence_length+1:end+1])
            targets.append(target_values[target_index])
            target_timestamps.append(timestamps.iloc[target_index])
            cities.append(city)
    return np.stack(sequences).astype(np.float32), np.asarray(targets, dtype=np.float32), pd.DataFrame({"Target_Timestamp": pd.to_datetime(target_timestamps), "City": cities})

if not TORCH_AVAILABLE:
    display(Markdown("PyTorch is not installed, so the optional neural-network model is skipped."))
elif DEEP_MODEL_KIND.startswith("gru"):
    sequence_features = (POLLUTION_COLUMNS if INCLUDE_EUROPEAN_AQI else NO_AQI_POLLUTION_COLUMNS) + TIME_FEATURES + LOCATION_FEATURES
    X_seq, y_seq, meta_seq = build_sequence_dataset(df_clean, sequence_features, TARGET_SOURCE, HORIZON_HOURS, SEQUENCE_LENGTH)
    masks = split_by_target_timestamp(meta_seq["Target_Timestamp"])
    X_seq_train, y_seq_train = X_seq[masks["train"].to_numpy()], y_seq[masks["train"].to_numpy()]
    X_seq_valid, y_seq_valid = X_seq[masks["valid"].to_numpy()], y_seq[masks["valid"].to_numpy()]
    X_seq_test, y_seq_test = X_seq[masks["test"].to_numpy()], y_seq[masks["test"].to_numpy()]
    meta_seq_test = meta_seq.loc[masks["test"].to_numpy()].reset_index(drop=True)
    if FAST_SMOKE_TEST:
        rng = np.random.default_rng(RANDOM_STATE)
        tr = rng.choice(len(X_seq_train), min(12000, len(X_seq_train)), replace=False); va = rng.choice(len(X_seq_valid), min(4000, len(X_seq_valid)), replace=False); te = rng.choice(len(X_seq_test), min(4000, len(X_seq_test)), replace=False)
        X_seq_train, y_seq_train = X_seq_train[tr], y_seq_train[tr]; X_seq_valid, y_seq_valid = X_seq_valid[va], y_seq_valid[va]; X_seq_test, y_seq_test = X_seq_test[te], y_seq_test[te]; meta_seq_test = meta_seq_test.iloc[te].reset_index(drop=True)
    scaler = StandardScaler().fit(X_seq_train.reshape(-1, X_seq_train.shape[-1]))
    X_seq_train = scaler.transform(X_seq_train.reshape(-1, X_seq_train.shape[-1])).reshape(X_seq_train.shape).astype(np.float32)
    X_seq_valid = scaler.transform(X_seq_valid.reshape(-1, X_seq_valid.shape[-1])).reshape(X_seq_valid.shape).astype(np.float32)
    X_seq_test = scaler.transform(X_seq_test.reshape(-1, X_seq_test.shape[-1])).reshape(X_seq_test.shape).astype(np.float32)
    class GRUModel(nn.Module):
        def __init__(self, input_size):
            super().__init__()
            hidden = 96 if not RUN_BALANCED_BACKUP else 48
            layers = 2 if not RUN_BALANCED_BACKUP else 1
            self.gru = nn.GRU(input_size=input_size, hidden_size=hidden, num_layers=layers, batch_first=True, dropout=0.15 if layers > 1 else 0.0)
            self.head = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, 1))
        def forward(self, x):
            _, hidden = self.gru(x)
            return self.head(hidden[-1]).squeeze(-1)
    model = GRUModel(X_seq_train.shape[-1]).to(DEVICE)
    batch_size = 1024 if DEVICE == "cuda" else 256
    epochs = 2 if FAST_SMOKE_TEST else (8 if RUN_BALANCED_BACKUP else 30)
    if TASK_TYPE == "classification":
        positives = max(float((y_seq_train == 1).sum()), 1.0); negatives = max(float((y_seq_train == 0).sum()), 1.0)
        train_target = y_seq_train
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([negatives / positives], device=DEVICE))
    else:
        target_scaler = StandardScaler().fit(y_seq_train.reshape(-1, 1))
        train_target = target_scaler.transform(y_seq_train.reshape(-1, 1)).ravel().astype(np.float32)
        loss_fn = nn.MSELoss()
    train_loader = DataLoader(TensorDataset(torch.tensor(X_seq_train), torch.tensor(train_target)), batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    history = []
    for epoch in range(1, epochs + 1):
        model.train(); losses = []
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad(); loss = loss_fn(model(batch_x), batch_y); loss.backward(); optimizer.step(); losses.append(float(loss.detach().cpu()))
        history.append({"epoch": epoch, "train_loss": float(np.mean(losses))})
    display(pd.DataFrame(history))
    model.eval()
    with torch.no_grad():
        valid_raw = model(torch.tensor(X_seq_valid, device=DEVICE)).detach().cpu().numpy()
        test_raw = model(torch.tensor(X_seq_test, device=DEVICE)).detach().cpu().numpy()
    if TASK_TYPE == "classification":
        valid_scores = 1 / (1 + np.exp(-valid_raw)); test_scores = 1 / (1 + np.exp(-test_raw))
        threshold = choose_threshold(pd.Series(y_seq_valid.astype(int)), valid_scores)
        valid_values, _ = classification_metrics(pd.Series(y_seq_valid.astype(int)), valid_scores, threshold); test_values, test_pred = classification_metrics(pd.Series(y_seq_test.astype(int)), test_scores, threshold)
        run_id = "PyTorch GRU sequence classifier"
        results.append(dict(run_id=run_id, model=run_id, params={"epochs": epochs, "sequence_length": SEQUENCE_LENGTH, "device": DEVICE}, threshold=threshold, **{f"valid_{k}": v for k, v in valid_values.items()}, **{f"test_{k}": v for k, v in test_values.items()}))
        prediction_tables[run_id] = meta_seq_test.assign(actual=y_seq_test.astype(int), predicted=test_pred, positive_score=test_scores)
    else:
        valid_pred = target_scaler.inverse_transform(valid_raw.reshape(-1, 1)).ravel(); test_pred = target_scaler.inverse_transform(test_raw.reshape(-1, 1)).ravel()
        valid_values = regression_metrics(pd.Series(y_seq_valid), valid_pred); test_values = regression_metrics(pd.Series(y_seq_test), test_pred)
        run_id = "PyTorch GRU sequence regressor"
        results.append(dict(run_id=run_id, model=run_id, params={"epochs": epochs, "sequence_length": SEQUENCE_LENGTH, "device": DEVICE}, **{f"valid_{k}": v for k, v in valid_values.items()}, **{f"test_{k}": v for k, v in test_values.items()}))
        prediction_tables[run_id] = meta_seq_test.assign(actual=y_seq_test, predicted=test_pred, error=test_pred-y_seq_test)
    results_df = pd.DataFrame(results)
    display(results_df.sort_values("valid_f1" if TASK_TYPE == "classification" else "valid_mae", ascending=(TASK_TYPE != "classification")))
elif DEEP_MODEL_KIND == "mlp_classifier":
    preprocess = make_preprocessor(feature_columns)
    X_train_dense = preprocess.fit_transform(X_train[feature_columns]).astype(np.float32); X_valid_dense = preprocess.transform(X_valid[feature_columns]).astype(np.float32); X_test_dense = preprocess.transform(X_test[feature_columns]).astype(np.float32)
    class MLPBinaryClassifier(nn.Module):
        def __init__(self, input_size):
            super().__init__(); hidden = 128 if not RUN_BALANCED_BACKUP else 64
            self.net = nn.Sequential(nn.Linear(input_size, hidden), nn.ReLU(), nn.Dropout(0.15), nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, 1))
        def forward(self, x):
            return self.net(x).squeeze(-1)
    model = MLPBinaryClassifier(X_train_dense.shape[1]).to(DEVICE)
    epochs = 2 if FAST_SMOKE_TEST else (8 if RUN_BALANCED_BACKUP else 30); batch_size = 1024 if DEVICE == "cuda" else 256
    y_train_array = y_train.to_numpy(dtype=np.float32); positives = max(float((y_train_array == 1).sum()), 1.0); negatives = max(float((y_train_array == 0).sum()), 1.0)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([negatives / positives], device=DEVICE)); optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    train_loader = DataLoader(TensorDataset(torch.tensor(X_train_dense), torch.tensor(y_train_array)), batch_size=batch_size, shuffle=True)
    history = []
    for epoch in range(1, epochs + 1):
        model.train(); losses = []
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad(); loss = loss_fn(model(batch_x), batch_y); loss.backward(); optimizer.step(); losses.append(float(loss.detach().cpu()))
        history.append({"epoch": epoch, "train_loss": float(np.mean(losses))})
    display(pd.DataFrame(history))
    model.eval()
    with torch.no_grad():
        valid_raw = model(torch.tensor(X_valid_dense, device=DEVICE)).detach().cpu().numpy(); test_raw = model(torch.tensor(X_test_dense, device=DEVICE)).detach().cpu().numpy()
    valid_scores = 1 / (1 + np.exp(-valid_raw)); test_scores = 1 / (1 + np.exp(-test_raw))
    threshold = choose_threshold(y_valid, valid_scores); valid_values, _ = classification_metrics(y_valid, valid_scores, threshold); test_values, test_pred = classification_metrics(y_test, test_scores, threshold)
    run_id = "PyTorch MLP classifier"
    results.append(dict(run_id=run_id, model=run_id, params={"epochs": epochs, "device": DEVICE}, threshold=threshold, **{f"valid_{k}": v for k, v in valid_values.items()}, **{f"test_{k}": v for k, v in test_values.items()}))
    prediction_tables[run_id] = meta_test.assign(actual=y_test.values, predicted=test_pred, positive_score=test_scores)
    results_df = pd.DataFrame(results)
    display(results_df.sort_values("valid_f1", ascending=False))

## Final Evaluation

The best model is selected using validation performance, then interpreted on the chronological test set.

In [ ]:
results_df = pd.DataFrame(results)
ranked_results = results_df.sort_values("valid_f1" if TASK_TYPE == "classification" else "valid_mae", ascending=(TASK_TYPE != "classification")).reset_index(drop=True)
display(ranked_results)
best_run_id = ranked_results.loc[0, "run_id"]
best_predictions = prediction_tables[best_run_id]
display(Markdown(f"### Best model selected by validation performance: `{best_run_id}`"))
display(best_predictions.head())
if TASK_TYPE == "classification":
    matrix = confusion_matrix(best_predictions["actual"], best_predictions["predicted"])
    fig, ax = plt.subplots(figsize=(5, 4)); sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", ax=ax); ax.set_title("Best model confusion matrix on test set"); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); plt.show()
    per_city = best_predictions.groupby("City").apply(lambda g: pd.Series(dict(rows=len(g), hazard_rate=g["actual"].mean(), precision=precision_score(g["actual"], g["predicted"], zero_division=0), recall=recall_score(g["actual"], g["predicted"], zero_division=0), f1=f1_score(g["actual"], g["predicted"], zero_division=0), false_negatives=int(((g["actual"] == 1) & (g["predicted"] == 0)).sum())))).sort_values("f1", ascending=False)
    display(per_city)
    fig, ax = plt.subplots(figsize=(10, 6)); per_city["f1"].sort_values().plot(kind="barh", ax=ax); ax.set_title("Per-city test F1 for best model"); plt.show()
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sample_plot = best_predictions.sample(min(len(best_predictions), 5000), random_state=RANDOM_STATE)
    sns.scatterplot(data=sample_plot, x="actual", y="predicted", alpha=0.35, ax=axes[0])
    low = min(sample_plot["actual"].min(), sample_plot["predicted"].min()); high = max(sample_plot["actual"].max(), sample_plot["predicted"].max())
    axes[0].plot([low, high], [low, high], color="black", linewidth=1); axes[0].set_title("Predicted vs actual on test set")
    sns.histplot(best_predictions["error"], bins=80, kde=True, ax=axes[1]); axes[1].set_title("Test residual distribution"); plt.show()
    per_city = best_predictions.groupby("City").apply(lambda g: pd.Series(dict(rows=len(g), mae=mean_absolute_error(g["actual"], g["predicted"]), rmse=math.sqrt(mean_squared_error(g["actual"], g["predicted"])), mean_actual=g["actual"].mean(), mean_error=g["error"].mean()))).sort_values("mae")
    display(per_city)
    fig, ax = plt.subplots(figsize=(10, 7)); per_city["mae"].sort_values().plot(kind="barh", ax=ax); ax.set_title("Per-city test MAE for best model"); plt.show()
best_sklearn_rows = ranked_results[ranked_results["run_id"].isin(trained_sklearn_models.keys())]
if len(best_sklearn_rows) > 0:
    importance_run_id = best_sklearn_rows.iloc[0]["run_id"]
    importance_model = trained_sklearn_models[importance_run_id]
    valid_for_importance = runtime_sample(splits["valid"], 2500, TASK_TYPE)
    scoring = "f1" if TASK_TYPE == "classification" else "neg_mean_absolute_error"
    importance = permutation_importance(importance_model, valid_for_importance["X"][feature_columns], valid_for_importance["y"], scoring=scoring, n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1)
    importance_df = pd.DataFrame({"feature": feature_columns, "importance_mean": importance.importances_mean, "importance_std": importance.importances_std}).sort_values("importance_mean", ascending=False).head(25)
    display(Markdown(f"### Permutation importance from `{importance_run_id}`"))
    display(importance_df)
    fig, ax = plt.subplots(figsize=(10, 8)); sns.barplot(data=importance_df, x="importance_mean", y="feature", ax=ax); ax.set_title("Top permutation importance features"); plt.show()

## Critical Analysis and Conclusion

This final section turns the technical outputs into assignment-ready interpretation, recommendations, limitations, and conclusion notes.

In [ ]:
if TASK_TYPE == "classification":
    best_metric_text = f"The selected model achieved test F1={ranked_results.loc[0, 'test_f1']:.3f}, recall={ranked_results.loc[0, 'test_recall']:.3f}, and precision={ranked_results.loc[0, 'test_precision']:.3f}."
    weaker_cities = per_city.sort_values("f1").head(5).index.tolist()
    recommendation = "For hazardous-event use cases, recall and false negatives matter because missed hazardous hours are operationally costly. Cities with weaker F1 or high false negatives should be reviewed before operational use."
else:
    best_metric_text = f"The selected model achieved test MAE={ranked_results.loc[0, 'test_mae']:.3f}, RMSE={ranked_results.loc[0, 'test_rmse']:.3f}, and R2={ranked_results.loc[0, 'test_r2']:.3f}."
    weaker_cities = per_city.sort_values("mae", ascending=False).head(5).index.tolist()
    recommendation = "For forecasting use cases, city-level error matters because a good global average can hide poor performance in specific cities. Cities with high MAE should be investigated for local pollution dynamics, extreme events, or missing external predictors."
cluster_counts = cluster_view["cluster"].value_counts().sort_index().to_dict()
display(Markdown(f"""
### Main finding

{best_metric_text}

### Interpretation

The chronological test period simulates future deployment better than a random split. The unsupervised K-Means section produced these city-cluster counts: `{cluster_counts}`, which helps connect model performance to pollution profiles.

### Cities needing extra review

The cities that need the most review under this variation are: `{weaker_cities}`.

### Recommendation

{recommendation}

### Limitations

- The dataset covers about one year, so it supports short-term forecasting/classification better than long-term seasonal forecasting.
- Weather, traffic, policy, wildfire, industrial, and sensor metadata are not included.
- For hazard classification, `European_AQI` is excluded from the primary feature set because it may leak the target definition.
- The PyTorch model depends on local CUDA/PyTorch availability; the scikit-learn models keep the notebook runnable without GPU packages.

### Conclusion

This notebook provides a complete standalone AML workflow: problem framing, EDA, preprocessing, chronological validation, multiple model comparisons, optional GPU neural modeling, unsupervised profiling, test-set evaluation, and practical interpretation.
"""))